In [17]:
import math
import pandas as pd
import numpy as np
import os
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.cuda.amp import autocast,GradScaler

In [18]:
def same_seed(seed):
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [19]:
device="cuda" if torch.cuda.is_available() else 'cpu'
print(device)
config={
    'seed':5201314,
    'valid_ratio':0.2,
    'n_epochs':24,
    'batch_size':128,
    'learning_rate':3e-4,
    'early_stop':5,
    'scheduler_patience':3,
    'scheduler_factor':0.5,
    'save_path':'./cancer_model/cancer_cnn_model.ckpt',
    'train_path':'E:/yvliu/train',
    'test_path':'E:/yvliu/test',
    'csv_path':'E:/yvliu/train_labels/train_labels.csv'
}

cuda


In [20]:
class cancer_dataset(Dataset):
    def __init__(self,df,path,transform=None):
        self.df=df
        self.path=path
        self.transform=transform

    def __getitem__(self, index):
        img_id=self.df.iloc[index,0]
        img_path=os.path.join(self.path,f'{img_id}.tif')
        img=Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img=self.transform(img)
        if len(self.df.columns)>1:
            label=self.df.iloc[index,1]
            return img,torch.tensor([label],dtype=torch.float32)
        else:
            return img,img_id

    def __len__(self):
        return len(self.df)

In [21]:
class res_block(nn.Module):
    def __init__(self,in_channels,out_channels,stride=1):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1)
        self.bn1=nn.BatchNorm2d(out_channels)
        self.relu=nn.ReLU()
        self.conv2=nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=stride,padding=1)
        self.bn2=nn.BatchNorm2d(out_channels)
        if in_channels!=out_channels or stride!=1:
            self.shortcut=nn.Sequential(
                nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut=nn.Sequential()

    def forward(self,x):
        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)
        out=self.conv2(out)
        out=self.bn2(out)
        out=out+self.shortcut(x)
        out=self.relu(out)
        return out

class cancer_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre_layer=nn.Sequential(
            nn.Conv2d(3,16,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.layer_1=res_block(in_channels=16,out_channels=16)
        self.pool_1=nn.MaxPool2d(kernel_size=2,stride=2)
        self.layer_2=res_block(in_channels=16,out_channels=32)
        self.pool_2=nn.MaxPool2d(kernel_size=2,stride=2)
        self.layer_3=res_block(in_channels=32,out_channels=32)
        self.layer_4=res_block(in_channels=32,out_channels=64)
        self.global_pool=nn.AdaptiveAvgPool2d((1,1))
        self.fc_layer=nn.Sequential(
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32,1)
        )

    def forward(self,x):
        x=self.pre_layer(x)
        x=self.layer_1(x)
        x=self.pool_1(x)
        x=self.layer_2(x)
        x=self.pool_2(x)
        x=self.layer_3(x)
        x=self.layer_4(x)
        x=self.global_pool(x)
        x=x.flatten(1)
        x=self.fc_layer(x)
        return x

In [22]:
def cancer_trainer(train_loader,valid_loader,config,model,device):
    criterion=nn.BCEWithLogitsLoss()
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=config['scheduler_factor'],patience=config['scheduler_patience'])
    if not os.path.isdir('./cancer_model'):
        os.mkdir('./cancer_model')
    n_epochs=config['n_epochs']
    best_loss=math.inf
    step=0
    early_stop_count=0
    scaler=torch.amp.GradScaler()

    for epoch in range(n_epochs):
        model.train()
        loss_record=[]
        train_pbar=tqdm(train_loader,position=0,leave=False)
        for x,y in train_pbar:
            x,y=x.to(device),y.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=device):
                pred=model(x)
                loss=criterion(pred,y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                step+=1
            loss_record.append(loss.detach().item())
            train_pbar.set_description(f'Epoch {epoch+1}/{n_epochs}')
        mean_train_loss=sum(loss_record)/len(loss_record)

        model.eval()
        loss_record=[]
        acc_record=[]
        for x,y in valid_loader:
            x,y=x.to(device),y.to(device)
            with torch.no_grad():
                pred=model(x)
                loss=criterion(pred,y)
                pred_class=(pred>0.0).float()
                acc=(pred_class==y).float().mean()
                loss_record.append(loss.item())
                acc_record.append(acc.item())
        mean_valid_loss=sum(loss_record)/len(loss_record)
        scheduler.step(mean_valid_loss)
        mean_valid_acc=sum(acc_record)/len(acc_record)
        print(f'Epoch: {epoch+1}/{n_epochs}, train loss: {mean_train_loss:.4f}, valid loss: {mean_valid_loss:.4f}, valid acc: {mean_valid_acc:.4f}')
        if mean_valid_loss<best_loss:
            best_loss=mean_valid_loss
            torch.save(model.state_dict(),config['save_path'])
            print(f'saving model with {best_loss:.3f}')
            early_stop_count=0
        else:
            early_stop_count+=1

        if early_stop_count>=config['early_stop']:
            print('\nmodel is not improving, so we halt the training session.')
            return


In [23]:
same_seed(config['seed'])
train_data=pd.read_csv(config['csv_path'])
test_id=sorted([os.path.splitext(f)[0]for f in os.listdir(config['test_path'])if f.endswith('.tif')])
test_data=pd.DataFrame({'id':test_id})
train_data,valid_data=train_test_split(train_data,test_size=config['valid_ratio'],random_state=config['seed'])
train_transform=transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=0,translate=(0.1,0.1),scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
base_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
train_dataset=cancer_dataset(train_data,config['train_path'],transform=train_transform)
valid_dataset=cancer_dataset(valid_data,config['train_path'],transform=base_transform)
test_dataset=cancer_dataset(test_data,config['test_path'],transform=base_transform)
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=config['batch_size'], shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, pin_memory=True)




In [24]:
model=cancer_model().to(device)
cancer_trainer(train_loader,valid_loader,config,model,device)

Epoch: 1/24, train loss: 0.3638, valid loss: 0.2698, valid acc: 0.8909
saving model with 0.270


Epoch: 2/24, train loss: 0.2764, valid loss: 0.2301, valid acc: 0.9090
saving model with 0.230


Epoch: 3/24, train loss: 0.2441, valid loss: 0.2409, valid acc: 0.9031


Epoch: 4/24, train loss: 0.2271, valid loss: 0.2327, valid acc: 0.9089


Epoch: 5/24, train loss: 0.2170, valid loss: 0.2094, valid acc: 0.9190
saving model with 0.209


...

Epoch: 16/24, train loss: 0.1647, valid loss: 0.1587, valid acc: 0.9419


Epoch: 17/24, train loss: 0.1622, valid loss: 0.1482, valid acc: 0.9458


Epoch: 18/24, train loss: 0.1595, valid loss: 0.1849, valid acc: 0.9317


Epoch: 19/24, train loss: 0.1588, valid loss: 0.1396, valid acc: 0.9494


Epoch: 20/24, train loss: 0.1476, valid loss: 0.1434, valid acc: 0.9460

model is not improving, so we halt the training session.


In [25]:
def predict(test_loader,model,device):
    model.eval()
    preds = []
    for x, _ in tqdm(test_loader):
        x=x.to(device)
        with torch.no_grad():
            pred=model(x)
            probability=torch.sigmoid(pred).squeeze(1)
            preds.append(probability.cpu().numpy())
    return np.concatenate(preds)

def save_pred(preds,test_id_list,file):
    with open(file,'w') as fp:
        writer=csv.writer(fp)
        writer.writerow(['id','label'])
        for i,p in zip(test_id_list,preds):
            writer.writerow([i,p])

model=cancer_model().to(device)
model.load_state_dict(torch.load(config['save_path']))
preds=predict(test_loader,model,device)
save_pred(preds,test_data['id'].tolist(),'cancel_submission.csv')

100%|██████████| 449/449 [05:39<00:00,  1.32it/s]
